In [ ]:
import os
from dotenv import load_dotenv

In [ ]:
load_dotenv()

In [ ]:
if not os.getenv("GOOGLE_API_KEY"):
    raise ValueError("Set GOOGLE_API_KEY in your environment/.env before running this notebook.")

In [ ]:
mem0_config = {
    "llm":{"provider": "gemini",
        "config": {
            "model": "gemini-3.6-flash",
            "api_key": os.environ["GOOGLE_API_KEY"],
            "temperature": 0,
            "max_tokens": 2000,   # ← ADD THIS LINE
        },},
    "embedder":{"provider": "huggingface",
        "config": {
            "model": "all-MiniLM-L6-v2",
        },},
    "vector_store": {
        "provider": "qdrant",
        "config": {
            "path": "./data/mem0-qdrant",
            "embedding_model_dims": 384,
        },}
}

In [ ]:
from mem0 import Memory

m = Memory.from_config(mem0_config)

In [ ]:
messages = [
    {"role": "user", "content": "Hi, I'm Alex. I love basketball and gaming."},
    {"role": "assistant", "content": "Hey Alex! I'll remember your interests."}
]
m.add(messages, user_id="alex")

In [ ]:
results = m.search("What are your hobbies?", filters={"user_id": "alex"})
print(results)

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os

In [ ]:
llm_gemini = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    temperature=0,
    max_tokens=128,
    google_api_key=os.environ["GOOGLE_API_KEY"],
)

llm_gemini.invoke("hi! i am aiswarya")

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool

In [ ]:
@tool
def dummy_tool(inputText:str)-> str:
    """Convert input text to uppercase. This is just for testing.
     Arguments:
        inputText {str} -- input text to be converted to uppercase
    """
    return f"Outputs : {inputText.upper()}"

In [ ]:
my_tools = [dummy_tool]

In [ ]:
my_agent = create_agent(model=llm_gemini,tools=my_tools)

In [ ]:
my_agent

In [ ]:
res = my_agent.invoke({"messages":[("user","hi! i am aiswarya")]})

In [ ]:
res_with_tool = my_agent.invoke({"messages":[("user","hi! i am yash and i want to convert `Ai agent` to uppercase")]})

In [ ]:
res_with_tool

In [ ]:
import sys
from pathlib import Path

# Add the project root to Python's path
project_root = Path("D:/AgenticAI_Projects/CustomerSupportAIAutomation").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Now the import works
from customer_support_agent.core.settings import get_settings

In [ ]:
import os

settings = get_settings()

print("GOOGLE_API_KEY present:", bool(settings.GOOGLE_API_KEY))
print("GOOGLE_API_KEY length:", len(settings.GOOGLE_API_KEY) if settings.GOOGLE_API_KEY else 0)
print("GOOGLE_API_KEY prefix:", (settings.GOOGLE_API_KEY or "")[:8])
print()
print("embedding model:", settings.effective_google_embedding_model)
print()

# Set the env var the way chroma_kb.py does
os.environ["GOOGLE_API_KEY"] = settings.GOOGLE_API_KEY
print("env var set:", bool(os.environ.get("GOOGLE_API_KEY")))
print()

# Now try the actual embedding function construction
from chromadb.utils import embedding_functions

try:
    fn = embedding_functions.GoogleGenaiEmbeddingFunction(
        model_name=settings.effective_google_embedding_model,
    )
    print("EMBEDDING FUNCTION CREATED OK")
except Exception as e:
    print("REAL ERROR:")
    print(type(e).__name__, ":", e)
    import traceback
    traceback.print_exc()

In [ ]:
import sys
sys.path.insert(0, "D:/AgenticAI_Projects/CustomerSupportAIAutomation")
print("Added project root to sys.path")
print("Python:", sys.executable)

In [ ]:
from customer_support_agent.core.settings import get_settings

settings = get_settings()
print("gemini_model:", settings.gemini_model)
print("google_api_key set:", bool(settings.GOOGLE_API_KEY))
print("mem0_top_k:", settings.mem0_top_k)
print("chroma_mem0_dir:", settings.chroma_mem0_dir)
print("chroma_mem0_path:", settings.chroma_mem0_path)

In [ ]:
from customer_support_agent.integrations.memory.mem0_store import CustomerMemoryStore
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model=settings.gemini_model,
    google_api_key=settings.GOOGLE_API_KEY,
    temperature=settings.llm_temperature,
)

store = CustomerMemoryStore(settings=settings, llm=llm)
print("Store created OK")

In [ ]:
print("=== prasad@gmail.com ===")
for m in store.list_memories(user_id='prasad@gmail.com', limit=50):
    print(' -', m.get('memory', '')[:100])

print()
print("=== company::tcs ===")
for m in store.list_memories(user_id='company::tcs', limit=50):
    print(' -', m.get('memory', '')[:100])

In [17]:
query = 'account closure charges'
print("Customer scope:")
for m in store.search(query=query, user_id='prasad@gmail.com', limit=10):
    print(' -', m.get('memory', '')[:80])
print()
print("Company scope:")
for m in store.search(query=query, user_id='company::tcs', limit=10):
    print(' -', m.get('memory', '')[:80])

2026-10-05 14:25:00.854 | INFO     | customer_support_agent.integrations.memory.mem0_store:search:72 - Searching memory for user_id: prasad@gmail.com with query: account closure charges


Customer scope:


2026-10-05 14:25:01.438 | INFO     | customer_support_agent.integrations.memory.mem0_store:search:72 - Searching memory for user_id: company::tcs with query: account closure charges



Company scope:


In [20]:
import requests
from customer_support_agent.core.settings import get_settings

settings = get_settings()

resp = requests.post(
    f"https://generativelanguage.googleapis.com/v1beta/models/{settings.gemini_model}:generateContent?key={settings.GOOGLE_API_KEY}",
    json={"contents": [{"parts": [{"text": "Say hi"}]}]},
    timeout=30,
)
print("Status:", resp.status_code)
print(resp.text[:300])

Status: 503
{
  "error": {
    "code": 503,
    "message": "This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.",
    "status": "UNAVAILABLE"
  }
}

